# Synthetic reservoir modelling and contour analysis

A reproducible quantitative geoscience demonstration built only from this repository’s fictional synthetic surface. It does not evaluate a real reservoir or imply hydrocarbon accumulation.

**Depth convention:** depth values are negative metres; larger numerical values are shallower. The crest is the maximum valid raw-grid depth. The grid-exit saddle is a domain-dependent maximum-bottleneck path result, not a geologically interpreted spill point.

Run this notebook from the repository root. It writes figures and CSVs to `examples/output/`.

## 1. Load and inspect the synthetic dataset

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "src" / "geoscience_toolkit.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from contour_analysis import analyze_grid, load_grid, write_outputs
from geoscience_toolkit import (
    plot_cross_sections, plot_robustness, plot_sensitivity, plot_structural_map,
    plot_surface_3d, robustness_analysis, sensitivity_analysis, write_report,
)

INPUT = ROOT / "data" / "synthetic" / "synthetic_surface.csv"
OUTPUT = ROOT / "examples" / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
xs, ys, grid = load_grid(INPUT)
print(f"Loaded {INPUT.name}: {len(xs)} x-nodes by {len(ys)} y-nodes")
print(f"Depth range: {np.nanmin(grid):.3f} to {np.nanmax(grid):.3f} m")

## 2. Data-quality checks

In [ ]:
valid = np.isfinite(grid)
dx, dy = float(np.diff(xs).mean()), float(np.diff(ys).mean())
quality = pd.DataFrame([{
    "nx_nodes": len(xs), "ny_nodes": len(ys),
    "spacing_x_m": dx, "spacing_y_m": dy,
    "valid_nodes": int(valid.sum()), "total_nodes": int(grid.size),
    "valid_node_coverage_pct": 100 * float(valid.mean()),
    "missing_depth_nodes": int((~valid).sum()),
}])
display(quality)
print("Grid coordinates are complete, unique, axis-aligned and regularly spaced; missing depth nodes remain masked.")

## 3. Visualize and quantify one selected structural closure

In [ ]:
threshold_m = float(np.nanmax(grid) - 60.0)
result = analyze_grid(xs, ys, grid, spill_depth_m=threshold_m, smoothing_sigma=1.0)
selected_metrics = pd.DataFrame([{
    "crest_x_m": result["crest_x_m"], "crest_y_m": result["crest_y_m"],
    "crest_depth_m": result["crest_depth_m"], "selected_threshold_m": result["spill_threshold_m"],
    "relief_m": result["relief_m"], "closure_area_m2": result["closure_area_m2"],
    "closure_area_km2": result["closure_area_km2"], "closure_status": result["closure_status"],
    "boundary_touched": result["boundary_touched"],
    "grid_spacing_x_m": result["grid_spacing_x_m"], "grid_spacing_y_m": result["grid_spacing_y_m"],
    "valid_data_coverage_pct": result["valid_data_coverage_pct"],
    "valid_mesh_coverage_pct": result["valid_mesh_coverage_pct"],
    "grid_spill_x_m": result["spill_x_m"], "grid_spill_y_m": result["spill_y_m"],
    "grid_spill_depth_m": result["grid_spill_depth_m"],
}])
display(selected_metrics)
print("Area is integrated by clipping valid piecewise-linear mesh triangles at the selected threshold.")

In [ ]:
map_path = plot_structural_map(result, OUTPUT / "structural_contours.png")
display(Image(filename=str(map_path)))

## 4. Spill-depth sensitivity

In [ ]:
sensitivity = sensitivity_analysis(xs, ys, grid, smoothing_sigma=1.0)
sensitivity.to_csv(OUTPUT / "spill_sensitivity.csv", index=False)
area_path, relief_path, scenarios_path = plot_sensitivity(sensitivity, OUTPUT, xs, ys, grid, 1.0)
display(sensitivity[["spill_depth_threshold_m", "relief_m", "closure_area_m2", "closure_area_km2", "closure_status", "boundary_touched"]])
display(Image(filename=str(area_path)))
display(Image(filename=str(relief_path)))
display(Image(filename=str(scenarios_path)))

## 5. Three-dimensional geometry and crest cross-sections

In [ ]:
surface_path = plot_surface_3d(result, OUTPUT / "surface_3d.png")
sections_path = plot_cross_sections(result, OUTPUT / "cross_sections.png")
display(Image(filename=str(surface_path)))
display(Image(filename=str(sections_path)))

## 6. Controlled robustness scenarios

In [ ]:
robustness = robustness_analysis(seed=20261026)
robustness.to_csv(OUTPUT / "robustness_scenarios.csv", index=False)
robustness_path = plot_robustness(robustness, OUTPUT / "robustness_comparison.png")
display(robustness[["experiment", "scenario", "closure_area_km2", "valid_data_coverage_pct", "valid_mesh_coverage_pct", "closure_status"]])
display(Image(filename=str(robustness_path)))

## 7. Export machine-readable results and a concise report

In [ ]:
legacy_map, legacy_metrics = write_outputs(result, OUTPUT, stem=INPUT.stem)
summary = {
    key: result[key] for key in (
        "crest_x_m", "crest_y_m", "crest_depth_m", "crest_smoothed_depth_m",
        "spill_threshold_m", "relief_m", "spill_x_m", "spill_y_m", "grid_spill_depth_m",
        "grid_exit_reachable", "closure_area_m2", "closure_area_km2", "has_closure",
        "closed_closure", "closure_status", "boundary_touched", "closure_node_count",
        "grid_nx", "grid_ny", "grid_spacing_x_m", "grid_spacing_y_m", "valid_node_count",
        "total_node_count", "valid_data_coverage_pct", "valid_mesh_coverage_pct", "missing_node_count",
    )
}
summary["smoothing_sigma_y_cells"] = 1.0
summary["smoothing_sigma_x_cells"] = 1.0
summary["input_file"] = INPUT.name
pd.DataFrame([summary]).to_csv(OUTPUT / "metrics_summary.csv", index=False)
report_path = write_report(result, sensitivity, robustness, OUTPUT, INPUT)
print(f"Report: {report_path.relative_to(ROOT)}")
print(f"Metrics: {(OUTPUT / 'metrics_summary.csv').relative_to(ROOT)}")
print(f"Sensitivity: {(OUTPUT / 'spill_sensitivity.csv').relative_to(ROOT)}")
print(f"Robustness: {(OUTPUT / 'robustness_scenarios.csv').relative_to(ROOT)}")
print("Interpretation: synthetic geometric closure only; not proof of hydrocarbons or a real reservoir evaluation.")